<a href="https://colab.research.google.com/github/wtraquinas/lab-tools-prompting/blob/main/solution_lab_tools_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run Ollama in Colab
---

[![5aharsh/collama](https://raw.githubusercontent.com/5aharsh/collama/main/assets/banner.png)](https://github.com/5aharsh/collama)

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around. This example uses [Llama 3.2](https://ollama.com/library/llama3.2) to generate a response from a prompted question using [LangChain Ollama Integration](https://python.langchain.com/docs/integrations/chat/ollama/).

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [6]:
import sys

!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
165 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [7]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull llama3.2`.

For other models check https://ollama.com/library

In [8]:
!ollama pull phi3

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

If you have a use-case that can help out others feel free to add your notebook to [Collama](https://github.com/5aharsh/collama/fork)

In [9]:
!pip install langchain-ollama

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="phi3")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What's the length of hypotenuse in a right angled triangle"})))

To find the length of the hypotenuse in a right-angled triangle, we need to use the Pythagorean theorem which states that for any right-angled triangle, the square of the length of its hypotenuse (the side opposite the right angle) is equal to the sum of squares of the other two sides. Let's assume you have a right-angled triangle with one leg measuring 'a' units and another leg measuring 'b' units in length.

Here are simple steps that would lead us to find the hypotenuse:

1) Square both lengths, i.e., calculate "a²" (the square of side a) and "b²" (the square of side b). 
2) Add these two squared numbers together; this will give you 'c²' where c is your unknown hypotenuse length. So the equation becomes: `c² = a² + b²`.
3) To find the value for ‘c’, take the square root of both sides which gives us √(a²+b²). This would be the exact formula to use when you want to calculate your triangle's hypotenuse length. Just substitute 'a' and 'b' with their respective lengths in units (like inches or cm).
4) Calculate it using a calculator, ensuring that all measurements are of the same unit before applying them into this theorem for correct results. 

Remember if you don’t have both side length values but one value and either angle information other than right angles in your triangle (like elevation or depression), trigonometric functions might be needed as well to find missing sides, which is a slightly different process!

<br>
<hr>
<br>
<br>

## 🚀 Your turn

Time to make this lab your own!

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information. In reality, if you're using more complex tools, you may start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

Replace the `multiply` and `add` tools with **3 new tools** of your choosing. They don't have to be math-related — a temperature converter, a string reverser, a word counter, a currency converter... pick whatever sounds fun to build.

<br>


Here's what to do:

1. **Design your 3 tools** — Write 3 new functions decorated with `@tool`, each with a clear docstring (this becomes the tool's description shown to the model) and typed arguments.
2. **Update the tools list** — Replace `tools = [multiply, add]` with your 3 new tools.
3. **Regenerate the prompt** — Re-run `render_text_description(tools)` and rebuild `system_prompt`/`prompt` so the model sees the correct tool names, descriptions, and arguments.
4. **Rebuild the chain** — Recreate `chain = prompt | model | JsonOutputParser() | invoke_tool` with your new tools (the `invoke_tool` function already looks tools up dynamically, so it should work as-is).
5. **Put it to the test** — Ask your chain several natural-language questions, one for each tool, and confirm it picks the right tool with the right arguments every time.
6. **Reflect** — In a markdown cell, briefly note any cases where the model picked the wrong tool or malformed the arguments, and why you think that happened.

<br>


💡 **Tip:**

- Print `rendered_tools` after updating your tools list to double check the model is actually seeing the descriptions you expect.

<br>


⭐️ **Bonus ideas:**

- Add a 4th tool that takes more than 2 arguments, or a mix of types (e.g. `str` and `int`), and see if the model still handles it correctly.
- Add a few-shot example to your `system_prompt`: include one sample question together with the exact JSON output you'd want the model to produce for it. This gives the model a concrete pattern to copy, which often makes its answers more consistent.
- Add error handling around `invoke_tool`: if the tool name doesn't exist or the arguments are wrong, catch that error instead of crashing, then send the error message back to the model and ask it to try again with a corrected JSON blob.
- Ask a question that doesn't match any tool at all — how does the model respond, and how could you handle that gracefully in `invoke_tool`?




---



1. Design your 3 tools
- Write 3 new functions decorated with @tool, each with a clear docstring (this becomes the tool's description shown to the model) and typed arguments.

In [13]:
from langchain_core.tools import tool

# -------------------------
# Temperature Converters
# -------------------------

@tool
def fahrenheit_to_celsius(fahrenheit: float) -> float:
    """
    Convert a temperature from degrees Fahrenheit (°F) to degrees Celsius (°C).
    """
    return (fahrenheit - 32) * 5 / 9


@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """
    Convert a temperature from degrees Celsius (°C) to degrees Fahrenheit (°F).
    """
    return (celsius * 9 / 5) + 32


# -------------------------
# Currency Converters
# -------------------------

@tool
def eur_to_usd(eur: float, exchange_rate: float = 1.17) -> float:
    """
    Convert an amount from Euros (EUR) to US Dollars (USD) using the provided exchange rate.
    """
    return eur * exchange_rate


@tool
def usd_to_eur(usd: float, exchange_rate: float = 1.17) -> float:
    """
    Convert an amount from US Dollars (USD) to Euros (EUR) using the provided exchange rate.
    """
    return usd / exchange_rate




2. Update the tools list
- Replace tools = [multiply, add] with your 3 new tools.

In [14]:
# List of available tools
tools = [
    fahrenheit_to_celsius,
    celsius_to_fahrenheit,
    eur_to_usd,
    usd_to_eur,
]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
fahrenheit_to_celsius
Convert a temperature from degrees Fahrenheit (°F) to degrees Celsius (°C).
{'fahrenheit': {'title': 'Fahrenheit', 'type': 'number'}}
--
celsius_to_fahrenheit
Convert a temperature from degrees Celsius (°C) to degrees Fahrenheit (°F).
{'celsius': {'title': 'Celsius', 'type': 'number'}}
--
eur_to_usd
Convert an amount from Euros (EUR) to US Dollars (USD) using the provided exchange rate.
{'eur': {'title': 'Eur', 'type': 'number'}, 'exchange_rate': {'default': 1.17, 'title': 'Exchange Rate', 'type': 'number'}}
--
usd_to_eur
Convert an amount from US Dollars (USD) to Euros (EUR) using the provided exchange rate.
{'usd': {'title': 'Usd', 'type': 'number'}, 'exchange_rate': {'default': 1.17, 'title': 'Exchange Rate', 'type': 'number'}}


3. Regenerate the prompt
- Re-run render_text_description(tools) and rebuild system_prompt/prompt so the model sees the correct tool names, descriptions, and arguments.

In [15]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)



fahrenheit_to_celsius(fahrenheit: float) -> float - Convert a temperature from degrees Fahrenheit (°F) to degrees Celsius (°C).
celsius_to_fahrenheit(celsius: float) -> float - Convert a temperature from degrees Celsius (°C) to degrees Fahrenheit (°F).
eur_to_usd(eur: float, exchange_rate: float = 1.17) -> float - Convert an amount from Euros (EUR) to US Dollars (USD) using the provided exchange rate.
usd_to_eur(usd: float, exchange_rate: float = 1.17) -> float - Convert an amount from US Dollars (USD) to Euros (EUR) using the provided exchange rate.


In [33]:
import json

# Define the example JSON as a Python dictionary
example_json_dict = {"name": "usd_to_eur", "arguments": {"usd": 100.0}}
# Convert the dictionary to a JSON string
json_string = json.dumps(example_json_dict)
# Escape curly braces for LangChain's template parser
escaped_json_string = json_string.replace('{', '{{').replace('}', '}}')

system_prompt = f"""
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.

Here's an example of how to respond:

User: How much is 100 US dollars in euros?
Assistant: ```json
{escaped_json_string}
```
"""

prompt = ChatPromptTemplate.from_messages(
    [('system', system_prompt), ('user', '{input}')]
)

4. Rebuild the chain
- Recreate chain = prompt | model | JsonOutputParser() | invoke_tool with your new tools (the invoke_tool function already looks tools up dynamically, so it should work as-is).

In [17]:
chain = prompt | model
message = chain.invoke({"input": "what's 32Celsius in Fahrenheit?"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

```json
{
  "name": "fahrenheit_to_celsius",
  "arguments": { }
}
```
In this case, since the user is asking for a conversion from Celsius to Fahrenheit and not using an amount of Euros or US Dollars nor providing their own exchange rate (the provided tool `eur_to_usd` was irrelevant), only the `fahrenheit_to_celsius` function name needs to be used. No arguments are required for this specific conversion request as there is no monetary value involved and we're assuming standard room temperature in Celsius, not Fahrenheit input provided by the user which would require an argument with "value" keys such as `fahrenheit_to_celsius`. 

So based on your question about converting 32 degrees Celsius to Fahrenheit:
```json
{
    "name": "celsius_to_fahrenheit",
    "arguments": {
        "celsius": 32.0
    }
}
```


5. Put it to the test
- Ask your chain several natural-language questions, one for each tool, and confirm it picks the right tool with the right arguments every time.

In [22]:
message = chain.invoke({"input": "what's 74 Fahrenheit in Celsius?"})
# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

```json
{
  "name": "fahrenheit_to_celsius",
  "arguments": {
    "fahrenheit": 74
  }
}
```

Here I used the `fahrenheit_to_celsius` tool, which converts Fahrenheit to Celsius. The user input of '74 degrees Fahrenheit' is mapped as an argument named `'fahrenheit'` with a value of `74`.


In [23]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "what's 74 Fahrenheit in Celsius?"})

{'name': 'fahrenheit_to_celsius', 'arguments': {'fahrenheit': 74}}

In [24]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

In [25]:
invoke_tool({"name": "eur_to_usd", "arguments": {"eur": 30}})

35.099999999999994

In [28]:
invoke_tool({"name": "usd_to_eur", "arguments": {"usd": 35}})

29.914529914529915

In [34]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "How many euros is 250 US dollars?"})

{'name': 'usd_to_eur',
 'arguments': {'usd': 250.0},
 'output': 213.67521367521368}

In [35]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "Convert 212 Fahrenheit to Celsius"})

{'name': 'fahrenheit_to_celsius',
 'arguments': {'fahrenheit': 212},
 'output': 100.0}

6. Reflect
- In a markdown cell, briefly note any cases where the model picked the wrong tool or malformed the arguments, and why you think that happened.

The model choose the correct model after being provided with examples.

With phi3 in Ollama, you cannot reliably demonstrate LangChain's automatic @tool selection because the model itself doesn't support function/tool calling in the way LangChain expects.

For a LangChain course or lab, these following models should be used instead of phi3:

- qwen2.5
- llama3.1
- OpenAI gpt-4.1 / gpt-4o / gpt-5

They support LangChain's tool-calling workflow much more reliably.